# Módulo 02 · Aula 2 — pandas: Series, DataFrames e seleção

**Capacitação Introdutória de Ciência de Dados · FEA.dev**

---

O **pandas** é a biblioteca que você vai usar todos os dias. Ele traz para o Python a
tabela: linhas, colunas com nome, tipos diferentes em cada coluna. É a planilha, com a
diferença de que cada passo fica registrado em código — reproduzível, auditável e
aplicável a milhões de linhas.

Ao final desta aula você vai saber:

- criar e entender uma **Series** e um **DataFrame**;
- **ler arquivos** CSV e Excel;
- inspecionar uma base que você nunca viu (`head`, `info`, `describe`);
- selecionar colunas, linhas e células com **`.loc` e `.iloc`** — sabendo qual usar;
- criar colunas novas e ordenar dados.

**Tempo estimado:** 75 minutos.

### Antes de começar — se você está no Google Colab

Este notebook lê arquivos da pasta `data/` do repositório, e no Colab a máquina começa vazia. **Execute a célula abaixo antes de qualquer outra**: ela traz o repositório e entra na pasta deste módulo, de modo que os caminhos `../data/...` usados no material funcionem sem alteração.

No VS Code ou no Jupyter local a célula não faz nada — os arquivos já estão no seu disco.

In [ ]:
# Setup do Google Colab.
# Traz o repositório da capacitação e entra na pasta deste módulo, para que os
# caminhos "../data/..." usados no material funcionem sem nenhuma alteração.
# Fora do Colab (VS Code, Jupyter local) esta célula não faz nada.
# Pode ser executada mais de uma vez sem problema.
import os
import subprocess
import sys

PASTA_DESTE_MODULO = "02_Manipulacao_Dados"
REPOSITORIO = "https://github.com/gustavokatsuo/Introducao-a-Ciencia-de-Dados.git"

if "google.colab" in sys.modules and not os.path.isdir("../data"):
    destino = "/content/Introducao-a-Ciencia-de-Dados"
    if not os.path.isdir(destino):
        print("Baixando o material da capacitação...")
        subprocess.run(["git", "clone", "--depth", "1", REPOSITORIO, destino], check=True)
    os.chdir(os.path.join(destino, PASTA_DESTE_MODULO))
    print("Pronto. Pasta de trabalho:", os.getcwd())

In [ ]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

print("pandas:", pd.__version__)

## 1. Series: uma coluna com rótulos

A `Series` é a estrutura de uma dimensão do pandas. Pense nela como um array do NumPy
que ganhou um **índice** — rótulos para cada posição.

In [ ]:
precos = pd.Series([32.50, 61.20, 28.75, 41.90])
precos

In [ ]:
# Repare nos números à esquerda: esse é o índice. E ele pode ser o que você quiser.
precos = pd.Series(
    [32.50, 61.20, 28.75, 41.90],
    index=["PETR4", "VALE3", "ITUB4", "WEGE3"],
    name="preco_fechamento",
)
precos

In [ ]:
print(precos["PETR4"])          # acesso pelo rótulo
print(precos.iloc[0])           # acesso pela posição
print(precos.mean().round(2))   # as agregações do NumPy continuam valendo
print(precos[precos > 35])      # e as máscaras booleanas também

## 2. DataFrame: a tabela

O `DataFrame` é um conjunto de Series que compartilham o mesmo índice. Ou seja: uma
tabela, em que cada coluna tem seu nome e seu tipo.

In [ ]:
from IPython.display import Image
Image("../assets/anatomia_dataframe.png", width=680)

Há várias formas de criar um. A mais comum na mão é a partir de um dicionário, em que
cada chave vira uma coluna:

In [ ]:
carteira = pd.DataFrame({
    "ticker": ["PETR4", "VALE3", "ITUB4", "WEGE3"],
    "preco": [32.50, 61.20, 28.75, 41.90],
    "quantidade": [100, 50, 200, 80],
    "setor": ["Petróleo e Gás", "Mineração", "Financeiro", "Bens industriais"],
})
carteira

In [ ]:
# E também a partir de uma lista de dicionários — exatamente a estrutura que
# construímos na aula 01.3, e a que as APIs costumam devolver.
registros = [
    {"ticker": "PETR4", "preco": 32.50, "quantidade": 100},
    {"ticker": "VALE3", "preco": 61.20, "quantidade": 50},
]
pd.DataFrame(registros)

## 3. Lendo arquivos

Na prática, você quase nunca digita os dados: você os lê de um arquivo.

Todos os dados desta capacitação estão na pasta `data/`, **dentro do repositório**.
Nenhum notebook baixa nada da internet enquanto roda. Como os notebooks estão em pastas
de módulo, o caminho relativo até os dados é sempre `../data/`.

In [ ]:
acoes = pd.read_csv("../data/acoes_b3.csv", parse_dates=["data"])
acoes.head()

Duas coisas aconteceram nessa linha:

- `read_csv` leu o arquivo e inferiu os tipos de cada coluna;
- `parse_dates=["data"]` pediu explicitamente para a coluna `data` virar **data de
  verdade** (`datetime`), não texto. Sem isso, `"2021-01-04"` seria só uma string, e
  você não conseguiria ordenar por mês, filtrar por ano ou calcular intervalos.

Outros parâmetros de `read_csv` que resolvem 90% dos problemas do mundo real:

| Parâmetro | Para quando |
|---|---|
| `sep=";"` | o arquivo usa ponto e vírgula (padrão do Excel em português) |
| `decimal=","` | os números usam vírgula decimal |
| `encoding="latin-1"` | aparecem caracteres estranhos no lugar dos acentos |
| `skiprows=3` | o arquivo tem linhas de cabeçalho antes da tabela |
| `na_values=["-", "N/A"]` | o arquivo marca ausência com um símbolo próprio |

In [ ]:
# Excel funciona igual — mas exige escolher a aba (sheet)
cotacoes_2025 = pd.read_excel("../data/acoes_2025.xlsx", sheet_name="cotacoes")
cadastro = pd.read_excel("../data/acoes_2025.xlsx", sheet_name="cadastro")

print("Abas lidas:", cotacoes_2025.shape, cadastro.shape)
cadastro

> **Dica:** Se você não sabe quais abas existem,
> `pd.ExcelFile("arquivo.xlsx").sheet_names` devolve a lista.

## 4. Inspecionando uma base nova

Este é o **primeiro** que se faz ao receber qualquer base — antes de qualquer análise.
Cinco comandos, nesta ordem.

In [ ]:
# 1. Qual o tamanho? (linhas, colunas)
acoes.shape

In [ ]:
# 2. Como são as primeiras linhas?
acoes.head(3)

In [ ]:
# 3. E as últimas? (revela se o arquivo tem lixo no fim, como totais ou notas de rodapé)
acoes.tail(3)

In [ ]:
# 4. Que colunas existem, de que tipo, e quantos valores faltam?
acoes.info()

`info()` é o comando mais informativo dos cinco. Leia com atenção:

- **`Non-Null Count`** — quantos valores não são nulos em cada coluna. Se um número
  aqui for menor que o total de linhas, aquela coluna tem buracos.
- **`Dtype`** — o tipo. `object` normalmente significa **texto**, e é aqui que moram os
  problemas: se uma coluna que deveria ser numérica aparece como `object`, tem número
  salvo como texto ali dentro.
- **`memory usage`** — o tamanho na memória.

In [ ]:
# 5. Como se distribuem os números?
acoes.describe()

`describe()` traz contagem, média, desvio padrão, mínimo, quartis e máximo de cada
coluna numérica. É a primeira leitura estatística da base — e serve como **detector de
absurdos**: preço mínimo negativo, idade máxima de 250 anos, volume zerado.

In [ ]:
# Alguns atributos úteis
print("Colunas:", list(acoes.columns))
print("Índice :", acoes.index)
print("Tipos  :")
print(acoes.dtypes)

In [ ]:
# Para colunas de texto, o que interessa são os valores distintos e a frequência
print(acoes["ticker"].unique())
print()
print(acoes["ticker"].value_counts())

## 5. Selecionando colunas

In [ ]:
# Uma coluna -> devolve uma Series
acoes["fechamento"].head()

In [ ]:
# Várias colunas -> devolve um DataFrame. Note os colchetes DUPLOS:
# o de fora seleciona; o de dentro é a lista de nomes.
acoes[["data", "ticker", "fechamento"]].head()

In [ ]:
print(type(acoes["fechamento"]))       # Series
print(type(acoes[["fechamento"]]))     # DataFrame com uma coluna só

> Existe também a forma `acoes.fechamento`, com ponto. Funciona, mas quebra quando o
> nome tem espaço, acento ou coincide com um método do pandas. **Use sempre colchetes** —
> é a forma que nunca falha.

## 6. `.loc` e `.iloc`: os dois seletores

Esta é a parte que mais confunde no começo, e vale investir dez minutos aqui.

O pandas tem **duas maneiras de endereçar** uma linha ou coluna:

- **`.loc`** usa **rótulos** (*labels*): o nome da coluna, o valor do índice;
- **`.iloc`** usa **posições** inteiras: 0, 1, 2... como em uma lista.

A sintaxe dos dois é `[linhas, colunas]` — igual ao NumPy em 2D.

In [ ]:
Image("../assets/loc_vs_iloc.png", width=900)

In [ ]:
# Vamos usar uma tabela pequena para enxergar o que acontece
amostra = acoes[acoes["ticker"] == "PETR4"].head(5).copy()
amostra = amostra[["data", "abertura", "fechamento", "volume"]]
amostra

Repare no índice à esquerda: `2`, `10`, `18`... Ele veio da tabela original e **não é**
0, 1, 2, 3. Isso é o que torna a distinção entre rótulo e posição concreta.

In [ ]:
# .iloc -> POSIÇÃO
print("Primeira linha, primeira coluna:", amostra.iloc[0, 0])
print("Segunda linha, terceira coluna :", amostra.iloc[1, 2])

In [ ]:
primeiro_rotulo = amostra.index[0]
print("Primeiro rótulo do índice:", primeiro_rotulo)

# .loc -> RÓTULO
print("Mesma célula, via rótulo:", amostra.loc[primeiro_rotulo, "abertura"])

In [ ]:
# Linhas inteiras
amostra.iloc[0]           # a primeira linha (posição)

In [ ]:
amostra.loc[primeiro_rotulo]    # a mesma linha, pelo rótulo

In [ ]:
# Fatias
print("--- .iloc[0:2] : posições 0 e 1, EXCLUI o 2 ---")
print(amostra.iloc[0:2])
print()
print("--- .loc com fatia de rótulos: INCLUI as duas pontas ---")
print(amostra.loc[amostra.index[0]:amostra.index[1]])

> **Atenção — Diferença que pega todo mundo:** `.iloc[0:2]` exclui o fim (dois elementos,
> como em listas), mas `.loc["a":"c"]` **inclui** as duas pontas. Faz sentido quando você
> pensa: com rótulos, o pandas não tem como saber "qual é o próximo", então ele inclui o
> que você nomeou.

In [ ]:
# Selecionando linhas e colunas ao mesmo tempo
print(amostra.iloc[0:3, 1:3])        # posições
print()
print(amostra.loc[:, ["data", "fechamento"]])   # todas as linhas, colunas nomeadas

### Quando usar cada um

| Situação | Use |
|---|---|
| "quero a coluna `fechamento`" | `.loc` |
| "quero as linhas onde o ticker é PETR4" | `.loc` (com máscara booleana) |
| "quero as 10 primeiras linhas, sem olhar o conteúdo" | `.iloc` |
| "quero a última linha, seja ela qual for" | `.iloc[-1]` |

Na prática, **`.loc` é o que você mais usa**, porque análise de dados é feita por nome
de coluna e por condição. `.iloc` fica para inspeção rápida e para percorrer posições.

E `.loc` também aceita máscaras booleanas — é assim que se filtra e se seleciona colunas
na mesma linha de código:

In [ ]:
acoes.loc[acoes["ticker"] == "WEGE3", ["data", "fechamento"]].head()

## 7. O índice

Por padrão o índice é 0, 1, 2... Mas ele pode ser qualquer coisa que identifique as
linhas — e escolher um índice significativo torna `.loc` muito mais expressivo.

In [ ]:
ibovespa = pd.read_csv("../data/ibovespa.csv", parse_dates=["data"])
print(ibovespa.head(3))

ibovespa_indexado = ibovespa.set_index("data")
ibovespa_indexado.head(3)

In [ ]:
# Com um índice de datas, .loc entende datas escritas como texto:
print(ibovespa_indexado.loc["2025-01-02"])
print()
print("--- todo o mês de janeiro de 2025 ---")
print(ibovespa_indexado.loc["2025-01"].head())

In [ ]:
# E permite fatiar por período:
janela = ibovespa_indexado.loc["2025-03-01":"2025-03-10", ["fechamento"]]
janela

In [ ]:
# reset_index desfaz: o índice volta a ser 0,1,2... e vira uma coluna comum
ibovespa_indexado.reset_index().head(3)

## 8. Criando e removendo colunas

Criar coluna é atribuir a um nome que ainda não existe. E, como no NumPy, a conta é
**vetorizada**: vale para a coluna inteira de uma vez, sem loop.

In [ ]:
carteira = pd.DataFrame({
    "ticker": ["PETR4", "VALE3", "ITUB4", "WEGE3"],
    "preco_compra": [30.10, 68.40, 26.90, 38.20],
    "preco_atual": [32.45, 61.20, 28.75, 41.90],
    "quantidade": [100, 50, 200, 80],
})

carteira["valor_investido"] = carteira["preco_compra"] * carteira["quantidade"]
carteira["valor_atual"] = carteira["preco_atual"] * carteira["quantidade"]
carteira["retorno"] = (carteira["preco_atual"] - carteira["preco_compra"]) / carteira["preco_compra"]

carteira

In [ ]:
# Colunas de texto usam o acessador .str, que aplica métodos de string à coluna inteira
carteira["ticker_minusculo"] = carteira["ticker"].str.lower()

# np.where para criar colunas condicionais
carteira["resultado"] = np.where(carteira["retorno"] > 0, "lucro", "prejuízo")

carteira[["ticker", "ticker_minusculo", "retorno", "resultado"]]

In [ ]:
# Para mais de duas faixas, pd.cut divide uma coluna numérica em intervalos nomeados
carteira["faixa"] = pd.cut(
    carteira["retorno"],
    bins=[-1, -0.05, 0.05, 1],
    labels=["queda", "estável", "alta"],
)
carteira[["ticker", "retorno", "faixa"]]

In [ ]:
# Removendo colunas: axis=1 significa "coluna" (axis=0 seria linha)
carteira = carteira.drop(columns=["ticker_minusculo"])
print(list(carteira.columns))

# Renomeando
carteira = carteira.rename(columns={"retorno": "retorno_pct"})
print(list(carteira.columns))

> **Dica:** Repare no padrão `carteira = carteira.drop(...)`. Quase todo método do pandas
> **devolve uma cópia modificada** em vez de alterar o original. Se você não reatribuir,
> nada muda. Existe o parâmetro `inplace=True`, mas ele está caindo em desuso — prefira
> reatribuir, é mais explícito e funciona sempre.

## 9. Ordenando

In [ ]:
carteira.sort_values("retorno_pct", ascending=False)[["ticker", "retorno_pct", "valor_atual"]]

In [ ]:
# Ordenando por mais de uma coluna: primeiro por ticker, depois por data
acoes.sort_values(["ticker", "data"], ascending=[True, False]).head()

In [ ]:
# nlargest / nsmallest: os N maiores, direto ao ponto
acoes.nlargest(5, "volume")[["data", "ticker", "fechamento", "volume"]]

## 10. Salvando o resultado

In [ ]:
from pathlib import Path
import tempfile

# Salvamos em uma pasta temporária apenas para demonstrar; no seu projeto,
# você escolheria um caminho de verdade.
destino = Path(tempfile.gettempdir()) / "carteira_analisada.csv"

carteira.to_csv(destino, index=False)   # index=False evita gravar a coluna 0,1,2...

conferencia = pd.read_csv(destino)
print(f"Arquivo salvo com {len(conferencia)} linhas e {conferencia.shape[1]} colunas")
conferencia.head(2)

## 11. Recapitulando

- **Series** = coluna com índice; **DataFrame** = conjunto de Series com o mesmo índice.
- `pd.read_csv(...)` e `pd.read_excel(..., sheet_name=...)` leem arquivos. Use
  `parse_dates` para datas; `sep`, `decimal` e `encoding` para arquivos brasileiros.
- Ao abrir uma base nova, sempre: `shape` → `head` → `tail` → `info` → `describe`.
- `df["col"]` devolve Series; `df[["a", "b"]]` devolve DataFrame.
- **`.loc` usa rótulos; `.iloc` usa posições.** Fatia com `.iloc` exclui o fim; com
  `.loc`, inclui. `.loc` também aceita máscaras booleanas.
- `set_index` / `reset_index` trocam o índice. Índice de datas permite
  `.loc["2025-01"]`.
- Criar coluna é atribuir; a conta é vetorizada. `np.where` e `pd.cut` criam colunas
  condicionais. `.str` aplica métodos de texto à coluna inteira.
- Métodos devolvem cópias: **reatribua** o resultado.

**Próxima aula:** filtros, agrupamentos e a construção de respostas a partir da tabela.